# MISO Environment Smoke Test
**Kernel: miso (Python 3.7)**

Tests the MISO environment using the Scanpy built-in Visium dataset.
No external data download required beyond the ~50MB Visium cache.
All plots saved to `./miso_smoke_test_plots/`.

> Note: Image feature extraction (ViT) is skipped in this smoke test as it takes
> 10min (GPU) to 2hrs (CPU). The pipeline is tested without the histology modality
> to verify core MISO functionality quickly.

In [1]:
# Cell 0 - WSL matplotlib setup (remove for Linux server, replace with %matplotlib inline)
import matplotlib
matplotlib.use('Agg')
matplotlib.rcParams['font.family'] = 'DejaVu Sans'

import matplotlib.pyplot as plt
import os

PLOT_DIR = './miso_smoke_test_plots'
os.makedirs(PLOT_DIR, exist_ok=True)

def save_plot(filename):
    path = os.path.join(PLOT_DIR, filename)
    plt.savefig(path, bbox_inches='tight', dpi=100)
    plt.close()
    print(f'  Plot saved: {path}')

print('✓ Matplotlib configured (Agg backend)')

✓ Matplotlib configured (Agg backend)


In [2]:
# Cell 1 - Version and import check
import sys
import torch
import scanpy as sc
import miso

print(f'Python:  {sys.version}')
print(f'torch:   {torch.__version__}')
print(f'scanpy:  {sc.__version__}')
print(f'miso:    {miso.__file__}')
print(f'CUDA available: {torch.cuda.is_available()}')
print(f'Device: {"GPU" if torch.cuda.is_available() else "CPU (training will be slower)"}')
print('\n✓ All MISO environment packages imported successfully')

Python:  3.7.16 (default, Jan 17 2023, 22:20:44) 
[GCC 11.2.0]
torch:   1.13.1+cu117
scanpy:  1.9.1
miso:    /home/jim/miniconda3/envs/miso_env/lib/python3.7/site-packages/miso/__init__.py
CUDA available: False
Device: CPU (training will be slower)

✓ All MISO environment packages imported successfully


In [3]:
# Cell 2 - Confirm pretrained ViT weights are present
# These are the LFS files downloaded via git lfs pull
import glob

miso_dir = os.path.dirname(miso.__file__)
pth_files = glob.glob(os.path.join(miso_dir, '**', '*.pth'), recursive=True)

print('Pretrained model weights:')
all_ok = True
for f in pth_files:
    size_mb = os.path.getsize(f) / (1024 * 1024)
    status = '✓' if size_mb > 1 else '✗ LFS STUB - run git lfs pull'
    print(f'  {status}  {os.path.basename(f)} ({size_mb:.1f} MB)')
    if size_mb < 1:
        all_ok = False

if not pth_files:
    print('  ✗ No .pth files found — check git lfs pull completed')
elif all_ok:
    print('\n✓ All pretrained weights downloaded correctly')
else:
    print('\n✗ Some weights are LFS stubs — run: git lfs pull')

Pretrained model weights:
  ✓  vit4k_xs_dino.pth (377.4 MB)
  ✓  vit256_small_dino.pth (671.6 MB)

✓ All pretrained weights downloaded correctly


In [4]:
# Cell 3 - Load Visium data and basic QC
# Uses the same built-in dataset as the spatial_main smoke test
# for easy cross-environment comparison
print('--- Loading Visium data ---')

import numpy as np

adata = sc.datasets.visium_sge(sample_id='V1_Human_Lymph_Node')
adata.var_names_make_unique()
print(f'  Loaded: {adata.shape[0]} spots x {adata.shape[1]} genes')

# Basic preprocessing
sc.pp.calculate_qc_metrics(adata, inplace=True)
sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)
sc.pp.highly_variable_genes(adata, n_top_genes=2000, flavor='seurat')
adata = adata[:, adata.var['highly_variable']].copy()
sc.pp.scale(adata)
print(f'  After HVG filter: {adata.shape}')

# Plot: QC violin
sc.pl.violin(adata, ['n_genes_by_counts', 'total_counts'], show=False)
save_plot('miso_qc_violin.png')

print('\n✓ Data loading and preprocessing OK')

--- Loading Visium data ---


/home/jim/miniconda3/envs/miso_env/lib/python3.7/site-packages/anndata/_core/anndata.py:1830: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")


  Loaded: 4035 spots x 36601 genes
  After HVG filter: (4035, 2000)
  Plot saved: ./miso_smoke_test_plots/miso_qc_violin.png

✓ Data loading and preprocessing OK


In [5]:
# Cell 4 - Build spatial graph (MISO input requirement)
print('--- Building spatial graph ---')

sc.pp.neighbors(adata, n_neighbors=6, use_rep='X')
print(f'  Spatial graph built: {adata.obsp["connectivities"].shape}')

# Extract spatial coordinates
spatial_coords = adata.obsm['spatial']
print(f'  Spatial coordinates shape: {spatial_coords.shape}')

# Plot: spatial coordinates
fig, ax = plt.subplots(figsize=(5, 5))
ax.scatter(spatial_coords[:, 0], spatial_coords[:, 1], s=2, alpha=0.5)
ax.set_title('Spatial coordinates')
ax.set_xlabel('x')
ax.set_ylabel('y')
save_plot('miso_spatial_coords.png')

print('\n✓ Spatial graph OK')

--- Building spatial graph ---


ModuleNotFoundError: No module named 'importlib.metadata'

In [12]:
import pandas as pd

# Cell 4b - OPTIONAL: H&E image feature extraction (ViT)
# WARNING: ~10 min on GPU, ~2 hrs on CPU
print('--- H&E image feature extraction ---')

from miso.hist_features import get_features, get_embeddings

# Check tissue image is available
if 'spatial' not in adata.uns:
    print('✗ No spatial image found in adata.uns["spatial"]')
else:
    library_id = list(adata.uns['spatial'].keys())[0]
    img = adata.uns['spatial'][library_id]['images']['hires']
    print(f'  H&E image shape: {img.shape}')
    print(f'  Library ID: {library_id}')

    scale_factor = adata.uns['spatial'][library_id]['scalefactors']['tissue_hires_scalef']
    spot_coords = adata.obsm['spatial'] * scale_factor
    print(f'  Spot coordinates scaled: {spot_coords.shape}')
    print(f'  Device: {"GPU" if torch.cuda.is_available() else "CPU — this will take ~2 hours"}')

    # Step 1: extract patch features from H&E image using ViT
    # locs needs to be a DataFrame with columns '4' and '5' as pixel coordinates
    locs_df = pd.DataFrame({
        '4': spot_coords[:, 0],  # x pixel coords
        '5': spot_coords[:, 1]   # y pixel coords
    }, index=adata.obs_names)

    print(locs_df.head())

    features = get_features(
        img=img,
        locs=locs_df,
        rad=100,
        pixel_size_raw=1.0,
        pixel_size=0.5,
        pretrained=True,
        device='cuda' if torch.cuda.is_available() else 'cpu'
    )

    adata.obsm['X_image'] = features
    print(f'  ✓ Spot image embeddings stored: {features.shape}')


    # Sanity plot: PCA of image embeddings in spatial coordinates
    from sklearn.decomposition import PCA
    pca = PCA(n_components=3)
    img_pca = pca.fit_transform(features)

    fig, axes = plt.subplots(1, 3, figsize=(12, 4))
    for i, ax in enumerate(axes):
        sc_plot = ax.scatter(
            spot_coords[:, 0], spot_coords[:, 1],
            c=img_pca[:, i], cmap='viridis', s=3
        )
        ax.set_title(f'Image PC{i+1}')
        ax.axis('off')
        plt.colorbar(sc_plot, ax=ax)
    plt.suptitle('PCA of H&E image embeddings (spatial distribution)')
    save_plot('miso_image_embeddings_pca.png')

    print('\n✓ Image feature extraction OK')
    print('  To use in MISO, update Cell 5 model init to include:')
    print('  image_features=adata.obsm["X_image"]')

--- H&E image feature extraction ---
  H&E image shape: (2000, 1921, 3)
  Library ID: V1_Human_Lymph_Node
  Spot coordinates scaled: (4035, 2)
  Device: CPU — this will take ~2 hours
                              4            5
AAACAAGTATCTCCCA-1  1419.749911  1187.717934
AAACAATCTACTAGCA-1   726.375763   231.861865
AAACACCAATAACTGC-1   448.243592  1373.479605
AAACAGAGCGACTCCT-1  1324.147293   454.537714
AAACAGCTTTCAGAAG-1   330.356378  1047.716236
Scaling image
Preprocessing image
Adjusting margins


Extracting image features: 1/4:   0%|          | 0/4 [00:00<?, ?it/s]

Extracting image features: 2/4:   0%|          | 0/4 [00:00<?, ?it/s]

Extracting image features: 3/4:   0%|          | 0/4 [00:00<?, ?it/s]

Extracting image features: 4/4:   0%|          | 0/4 [00:00<?, ?it/s]

Smoothing embeddings
  ✓ Spot image embeddings stored: (4035, 576)
  Plot saved: ./miso_smoke_test_plots/miso_image_embeddings_pca.png

✓ Image feature extraction OK
  To use in MISO, update Cell 5 model init to include:
  image_features=adata.obsm["X_image"]


In [19]:
import inspect
from miso import Miso
print(inspect.signature(Miso.__init__))

(self, features, ind_views='all', combs='all', sparse=False, neighbors=None, device='cpu')


In [21]:
# Cell 5 - MISO model: transcriptomics + image pipeline
print('--- MISO model pipeline (ST + histology) ---')

import scipy.sparse as sp
from miso import Miso

# Prepare transcriptomics matrix
X = adata.X if not sp.issparse(adata.X) else adata.X.toarray()
print(f'  Expression matrix: {X.shape}')

# Prepare image features
X_image = adata.obsm['X_image']
print(f'  Image feature matrix: {X_image.shape}')

# Prepare spatial neighbors as adjacency matrix
sc.pp.neighbors(adata, n_neighbors=6, use_rep='X')
adj = adata.obsp['connectivities']
print(f'  Adjacency matrix: {adj.shape}')

# Initialise Miso with both modalities as a list of feature matrices
model = Miso(
    features=[X, X_image],   # list of modality matrices
    neighbors=adj,            # spatial adjacency
    device='cuda' if torch.cuda.is_available() else 'cpu'
)
print('  ✓ Miso model initialised with 2 modalities')

# Check train signature before calling
import inspect
print(inspect.signature(model.train))

--- MISO model pipeline (ST + histology) ---
  Expression matrix: (4035, 2000)
  Image feature matrix: (4035, 576)
  Adjacency matrix: (4035, 4035)
  ✓ Miso model initialised with 2 modalities
()


In [23]:
import inspect
print(inspect.signature(model.train))
print(inspect.signature(model.get_embedding) if hasattr(model, 'get_embedding') else 'no get_embedding')
print([m for m in dir(model) if not m.startswith('_')])

()
no get_embedding
['T_destination', 'add_module', 'adj', 'adj1', 'apply', 'bfloat16', 'buffers', 'children', 'cluster', 'combinations', 'cpu', 'cuda', 'device', 'double', 'dump_patches', 'eval', 'extra_repr', 'features', 'float', 'forward', 'get_buffer', 'get_extra_state', 'get_parameter', 'get_submodule', 'half', 'ind_views', 'ipu', 'load_state_dict', 'modules', 'named_buffers', 'named_children', 'named_modules', 'named_parameters', 'num_views', 'parameters', 'pcs', 'register_backward_hook', 'register_buffer', 'register_forward_hook', 'register_forward_pre_hook', 'register_full_backward_hook', 'register_load_state_dict_post_hook', 'register_module', 'register_parameter', 'requires_grad_', 'set_extra_state', 'share_memory', 'sparse', 'state_dict', 'to', 'to_empty', 'train', 'training', 'type', 'xpu', 'zero_grad']


In [25]:
# Cell 6 - Train, extract embeddings and cluster
print('--- Training MISO model ---')

# Train
model.train()
print('  ✓ Training complete')

print(f'pcs type: {type(model.pcs)}')
print(f'pcs length: {len(model.pcs)}')

for i, item in enumerate(model.pcs):
    if hasattr(item, 'shape'):
        print(f'  pcs[{i}]: {type(item)} shape={item.shape}')
    else:
        print(f'  pcs[{i}]: {type(item)} value={item}')

# Get embedding via pcs (principal components of latent space)
embedding = model.pcs
print(f'  Embedding type: {type(embedding)}')

# Convert to numpy if needed
import numpy as np
if hasattr(embedding, 'detach'):
    embedding = embedding.detach().cpu().numpy()
elif not isinstance(embedding, np.ndarray):
    embedding = np.array(embedding)



--- Training MISO model ---


Training network for modality 1:   0%|          | 0/1000 [00:00<?, ?it/s]

Training network for modality 2:   0%|          | 0/1000 [00:00<?, ?it/s]

  ✓ Training complete
pcs type: <class 'list'>
pcs length: 2
  pcs[0]: <class 'torch.Tensor'> shape=torch.Size([4035, 128])
  pcs[1]: <class 'torch.Tensor'> shape=torch.Size([4035, 128])
  Embedding type: <class 'list'>


/home/jim/miniconda3/envs/miso_env/lib/python3.7/site-packages/ipykernel_launcher.py:26: FutureWarning: The input object of type 'Tensor' is an array-like implementing one of the corresponding protocols (`__array__`, `__array_interface__` or `__array_struct__`); but not a sequence (or 0-D). In the future, this object will be coerced as if it was first converted using `np.array(obj)`. To retain the old behaviour, you have to either modify the type 'Tensor', or assign to an empty array created with `np.empty(correct_shape, dtype=object)`.
/home/jim/miniconda3/envs/miso_env/lib/python3.7/site-packages/ipykernel_launcher.py:26: VisibleDeprecationWarning: Creating an ndarray from ragged nested sequences (which is a list-or-tuple of lists-or-tuples-or ndarrays with different lengths or shapes) is deprecated. If you meant to do this, you must specify 'dtype=object' when creating the ndarray.


In [ ]:
print(f'  ✓ Embedding shape: {embedding.shape}')
adata.obsm['X_miso'] = embedding

# Cluster
print('--- Clustering ---')
import inspect
print(inspect.signature(model.cluster))

In [ ]:
# Cell 7 - Summary
import glob as glob_mod

plots = glob_mod.glob(os.path.join(PLOT_DIR, '*.png'))
print('=== MISO Smoke Test Complete ===')
print(f'Plots saved to: {PLOT_DIR}/')
for p in sorted(plots):
    size_kb = os.path.getsize(p) / 1024
    print(f'  {os.path.basename(p)} ({size_kb:.1f} KB)')

print('\nNext step for full pipeline:')
print('  See official tutorial at:')
print('  https://github.com/kpcoleman/miso/blob/main/tutorial/tutorial.ipynb')
print('  (requires ~1GB tutorial data download + GPU for image features)')